#Imaginemos que tenemos una imagen suavizada, y sabemos el kernel Gaussiano que se empleó para obtenerla. ¿Es posible recuperar exactamente la imagen original a partir de dicha imagen suavizada y el kernel Gaussiano empleado?

La respuesta es que, en general, NO.

A mi juicio, hay dos razonamientos principales que nos llevan a concluir que, en el caso general, la recuperación de la imagen original no es posible. El primero está vinculado al dominio del espacio (que es lo que llevamos empleando desde el comienzo de la asignatura). El segundo está relacionado con el dominio de la frecuencia (que veremos más adelante en clases de teoría y que, en el momento actual, os puede resultar un poco confuso; pero, en cualquier caso, me parecía adecuado incluirlo).

#1. DOMINIO DEL ESPACIO

##1.1 CASO 1D

A nivel espacial, el resultado del suavizado, sea con una Gaussiana o con un box filter, implica una media ponderada de valores, en donde no hay información posicional. De modo que sabemos el resultado final y cómo se ponderaron los píxeles originales (en caso de conocer el kernel empleado), pero nada más que eso: no sabemos dónde estaba el píxel original ni de qué modo exacto se compensó ese valor con el de los vecinos. Por ejemplo, supongamos el siguiente kernel (1D por sencillez):
$\frac{1}{4}[1, 2, 1] = [\frac{1}{4}, \frac{1}{2}, \frac{1}{4}]$ y una serie de valores calculados 1D: `[12 26 38 32 20]`. Tendríamos:

$$
\begin{align*}
\frac{1}{4}a + \frac{1}{2}b + \frac{1}{4}c &= 12 \\
\frac{1}{4}b + \frac{1}{2}c + \frac{1}{4}d &= 26 \\
\frac{1}{4}c + \frac{1}{2}d + \frac{1}{4}e &= 38 \\
\frac{1}{4}d + \frac{1}{2}e + \frac{1}{4}f &= 32 \\
\frac{1}{4}e + \frac{1}{2}f + \frac{1}{4}g &= 20
\end{align*}
$$

En donde `[a, b, c, d, e, f, g]` son los valores originales a recuperar. El problema es que este sistema es indeterminado: tenemos 7 variables/incógnitas y 5 ecuaciones. Hay infinitas soluciones que verifican ese sistema.

Ahora, acotemos el problema con más información "privilegiada" (que podríamos no tener en absoluto, al igual que podríamos no tener el kernel exacto de suavizado empleado):
* _a, b, c, d, e, f, g_ deben ser enteros entre 0 y 255. No valen números negativos. Asumimos que la matriz de entrada nunca va a ser una imagen de derivadas, ni va a contener valores negativos o superiores a 255 (cosa que, insisto, no tiene por qué ser cierta).
* _a_ y _g_ se corresponden con el padding y sabemos que se han usado bordes replicados.

Todo esto proporcionaría el siguiente sistema de ecuaciones:
$$
\begin{align*}
\frac{1}{4}b + \frac{1}{2}b + \frac{1}{4}c &= 12 \\
\frac{1}{4}b + \frac{1}{2}c + \frac{1}{4}d &= 26 \\
\frac{1}{4}c + \frac{1}{2}d + \frac{1}{4}e &= 38 \\
\frac{1}{4}d + \frac{1}{2}e + \frac{1}{4}f &= 32 \\
\frac{1}{4}e + \frac{1}{2}f + \frac{1}{4}f &= 20
\end{align*}
$$
$$
\begin{align*}
0 \le b \le 255 \\
0 \le c \le 255 \\
0 \le d \le 255 \\
0 \le e \le 255 \\
0 \le f \le 255
\end{align*}
$$

Con esta información ya tenemos solución única: `(b,c,d,e,f)=(8,24,48,32,16)`

Pero fijémonos que disponemos de más información de la que, en general, tendríamos en un caso real:
* sabemos que la matriz/vector original contenía exclusivamente valores enteros entre 0 y 255.
* sabemos el tipo de padding realizado durante la convolución.
* sabemos exactamente el kernel empleado durante la convolución.
* asumimos que no hay nada de ruido, ni limitaciones de precisión o redondeo, en ninguna parte del proceso.

Si, por ejemplo, no disponemos de la información sobre el padding, tendríamos:
$$
\begin{align*}
\frac{1}{4}a + \frac{1}{2}b + \frac{1}{4}c &= 12 \\
\frac{1}{4}b + \frac{1}{2}c + \frac{1}{4}d &= 26 \\
\frac{1}{4}c + \frac{1}{2}d + \frac{1}{4}e &= 38 \\
\frac{1}{4}d + \frac{1}{2}e + \frac{1}{4}f &= 32 \\
\frac{1}{4}e + \frac{1}{2}f + \frac{1}{4}g &= 20
\end{align*}
$$
$$
\begin{align*}
0 \le a \le 255 \\
0 \le b \le 255 \\
0 \le c \le 255 \\
0 \le d \le 255 \\
0 \le e \le 255 \\
0 \le f \le 255 \\
0 \le g \le 255
\end{align*}
$$
Y esto, de nuevo, proporcionaría más de una solución válida. Si asumimos valores enteros, entre las posibles soluciones (compatibles con esas ecuaciones) podríamos citar las siguientes:
* `(a,b,c,d,e,f,g)=(8,8,24,48,32,16,16)`
* `(a,b,c,d,e,f,g)=(8,10,20,54,24,26,4)`
* `(a,b,c,d,e,f,g)=(20,0,28,48,28,24,4)`
* ...

Si asumiésemos valores en punto flotante (algo perfectamente posible al operar con imágenes) el número de soluciones posibles se vuelve infinito.

##1.2 CASO 2D

###1.2.1 Salida compuesta por un único píxel

Asumamos que queremos recuperar la imagen $3\times3$ a partir de la cual hemos, empleando un kernel $3\times3$ conocido, generado un nuevo píxel.

La imagen original, desconocida, sería:
$$
X_{\text{original}} = \begin{pmatrix}
x_{11} & x_{12} & x_{13} \\
x_{21} & x_{22} & x_{23} \\
x_{31} & x_{32} & x_{33}
\end{pmatrix}
$$

El kernel conocido sería:
$$
K_{\text{conocido}} = \begin{pmatrix}
\frac{1}{16} & \frac{1}{8} & \frac{1}{16} \\
\frac{1}{8} & \frac{1}{4} & \frac{1}{8} \\
\frac{1}{16} & \frac{1}{8} & \frac{1}{16}
\end{pmatrix}
$$

El valor del píxel de salida también lo conocemos: $5$

Todo lo anterior nos proporcionaría la siguiente ecuación:
$$
\begin{split}
\frac{1}{16}x_{11} + \frac{1}{8}x_{12} + \frac{1}{16}x_{13} \ + \\
\frac{1}{8}x_{21} + \frac{1}{4}x_{22} + \frac{1}{8}x_{23} \ + \\
\frac{1}{16}x_{31} + \frac{1}{8}x_{32} + \frac{1}{16}x_{33} = 5
\end{split}
$$

En este caso concreto, por ejemplo, la imagen original podría ser:
$$
\begin{pmatrix}
1 & 2 & 3 \\
4 & 5 & 6 \\
7 & 8 & 9
\end{pmatrix}
$$

Aunque, evidentemente, no es posible recuperarla, dado que tenemos una ecuación para $9$ incógnitas. Aún asumiendo valores enteros entre $0$ y $255$ (cosa que, como decía, no tiene por qué cumplirse, porque podemos intentar suavizar, por ejemplo, el resultado de una operación de derivadas o cualquier otra matriz/imagen fruto de operaciones previas), habría un conjunto enorme (pero finito) de posibles soluciones.

###1.2.2 Salida compuesta por múltiples píxeles

Ahora imaginemos este mismo ejemplo pero aumentando el número de píxeles generados de que disponemos.

Imaginemos una imagen original desconocida de $4\times4$:
$$
X_{\text{original}} = \begin{pmatrix}
x_{11} & x_{12} & x_{13} & x_{14} \\
x_{21} & x_{22} & x_{23} & x_{24} \\
x_{31} & x_{32} & x_{33} & x_{34} \\
x_{41} & x_{42} & x_{43} & x_{44}
\end{pmatrix}
$$

Tenemos el mismo kernel $K$ de antes ($3\times3$). La imagen de salida $G$, en este caso, sería de $2\times2$. Para generar $2\times2$ píxeles de salida a partir de una entrada $4\times4$ con un kernel $3\times3$, se debe realizar una convolución "válida" (sin padding). Asumamos que la salida fue:
$$
G_{\text{salida}} = \begin{pmatrix}
10 & 20 \\
30 & 40
\end{pmatrix}
$$

Cada píxel de la salida nos da una ecuación (para simplificar, multiplicaremos las ecuaciones por $16$):
$$
\begin{align*}
1x_{11} + 2x_{12} + 1x_{13} + 2x_{21} + 4x_{22} + 2x_{23} + 1x_{31} + 2x_{32} + 1x_{33} &= 160 \\
1x_{12} + 2x_{13} + 1x_{14} + 2x_{22} + 4x_{23} + 2x_{24} + 1x_{32} + 2x_{33} + 1x_{34} &= 320 \\
1x_{21} + 2x_{22} + 1x_{23} + 2x_{31} + 4x_{32} + 2x_{33} + 1x_{41} + 2x_{42} + 1x_{43} &= 480 \\
1x_{22} + 2x_{23} + 1x_{24} + 2x_{32} + 4x_{33} + 2x_{34} + 1x_{42} + 2x_{43} + 1x_{44} &= 640
\end{align*}
$$

Tenemos $16$ incógnitas y solo $4$ ecuaciones. El sistema es indeterminado.

Ahora bien, si decidimos obtener una salida de $4\times4$ (mismo tamaño que la entrada desconocida) y conocemos el padding, la situación cambia: seguimos teniendo $16$ incógnitas, pero ahora tenemos $16$ ecuaciones. Pero, como comentaba antes, estamos introduciendo restricciones para poder recuperar la imagen original. En el caso general, sigue sin ser posible.

###1.2.3 Nota sobre invertibilidad de la matriz de convolución

La convolución se puede plantear de forma algebraica como un producto de matrices: $Y = M \cdot X$, siendo $Y$ la imagen resultante en formato vector columna, $M$ una matriz construida a partir de los coeficientes del kernel, y $X$ la imagen de entrada también en formato vectorial.

Por ejemplo, si la imagen de entrada fuese de $2\times2$:
$$
X = \begin{pmatrix}
x_{11} & x_{12} \\
x_{21} & x_{22}
\end{pmatrix}
$$
Y el kernel fuese de $3\times3$ (pero, por sencillez, obviamos la constante de normalización $\frac{1}{16}$):
$$
K = \begin{pmatrix}
1 & 2 & 1 \\
2 & 4 & 2 \\
1 & 2 & 1
\end{pmatrix}
$$
La salida, conocida, sería otra matriz de $2\times2$:
$$
Y = \begin{pmatrix}
y_{11} & y_{12} \\
y_{21} & y_{22}
\end{pmatrix}
$$
Las $4$ ecuaciones que relacionan la salida con la entrada son (asumiendo $0$-padding):
$$
\begin{align*}
y_{11} &= 4x_{11} + 2x_{12} + 2x_{21} + 1x_{22} \\
y_{12} &= 2x_{11} + 4x_{12} + 1x_{21} + 2x_{22} \\
y_{21} &= 2x_{11} + 1x_{12} + 4x_{21} + 2x_{22} \\
y_{22} &= 1x_{11} + 2x_{12} + 2x_{21} + 4x_{22}
\end{align*}
$$
La matriz de convolución $M$ de $4\times4$ para este sistema es:
$$
M = \begin{pmatrix}
4 & 2 & 2 & 1 \\
2 & 4 & 1 & 2 \\
2 & 1 & 4 & 2 \\
1 & 2 & 2 & 4
\end{pmatrix}
$$
Y el sistema quedaría:
$$ Y = M \cdot X $$
$$
\begin{pmatrix} y_{11} \\ y_{12} \\ y_{21} \\ y_{22} \end{pmatrix} =
\begin{pmatrix}
4 & 2 & 2 & 1 \\
2 & 4 & 1 & 2 \\
2 & 1 & 4 & 2 \\
1 & 2 & 2 & 4
\end{pmatrix}
\cdot
\begin{pmatrix} x_{11} \\ x_{12} \\ x_{21} \\ x_{22} \end{pmatrix}
$$

La cuestión clave aquí es que, para recuperar $X$, la matriz $M$ debe ser invertible. Y no siempre lo es. Si no es invertible, no podemos recuperar $X$ a partir de $M^{-1} \cdot Y$. De hecho, si $M$ está muy **mal condicionada**, es decir, si el determinante de $M$ es muy próximo a cero, estaríamos amplificando brutalmente el ruido (como ocurrirá cuando hablemos del dominio de la frecuencia). Tendríamos $$ X = M^{-1} \cdot Y = \frac{1}{\det(M)} \cdot \text{Adj}(M) \cdot Y$$ La parte más importante de esta expresión matemática es el término $1 / det(M)$. Si $det(M)$ es un número muy pequeño, como $10^{-18}$, entonces, $1 / det(M)$ es un número gigantesco, $10^{18}$. Esto significa que los valores en la matriz inversa $M^{-1}$ son enormes.





Por ejemplo, imaginémonos que el kernel de suavizado es un box filter con todo unos. $M$ sería (obviando la constante de normalización):
$$
M_{\text{box}} = \begin{pmatrix}
1 & 1 & 1 & 1 \\
1 & 1 & 1 & 1 \\
1 & 1 & 1 & 1 \\
1 & 1 & 1 & 1
\end{pmatrix}
$$
Y esta matriz, evidentemente, tiene determinante cero y, como consecuencia, no es invertible.

Uno de los principales problemas es que, cuanto más grande sea la imagen ($M$ de mayor tamaño), en general, más probable es que la matriz sea mal condicionada (aunque no necesariamente singular). Esto, como decíamos, dificulta, en el caso general, recuperar la imagen original a partir del kernel y la imagen resultante.

Véase la siguiente implementación (refinada a partir del código proporcionado por ChatGPT), en donde se ve que, a mayor tamaño de $M$, menor valor del determinante y mayor el *condition number* (véase https://en.wikipedia.org/wiki/Condition_number):

In [ ]:
import numpy as np

# Creamos la matriz M
def create_convolution_matrix(image_shape, kernel):
    H, W = image_shape
    k_h, k_w = kernel.shape
    num_pixels = H * W
    M = np.zeros((num_pixels, num_pixels))
    k_center_y, k_center_x = k_h // 2, k_w // 2
    for r_out in range(H):
        for c_out in range(W):
            output_pixel_idx = r_out * W + c_out
            for r_k in range(k_h):
                for c_k in range(k_w):
                    r_in = r_out + r_k - k_center_y
                    c_in = c_out + c_k - k_center_x
                    if 0 <= r_in < H and 0 <= c_in < W:
                        input_pixel_idx = r_in * W + c_in
                        M[output_pixel_idx, input_pixel_idx] = kernel[r_k, c_k]
    return M

# --- Kernels a comparar ---
kernel_unnormalized = np.array([[1,2,1],[2,4,2],[1,2,1]])
kernel_normalized = kernel_unnormalized / np.sum(kernel_unnormalized)

# --- Análisis ---
image_sizes = [2, 3, 4, 5, 25, 50]

for normalize in [False, True]:
    if normalize:
        kernel = kernel_normalized
        print("\n--- ANÁLISIS CON KERNEL NORMALIZADO (Suma = 1) ---")
    else:
        kernel = kernel_unnormalized
        print("\n--- ANÁLISIS CON KERNEL NO NORMALIZADO (Suma = 16) ---")

    print(f"{'Tamaño Imagen':<15} {'Determinante':<15} {'Nº Condición':<15} ")
    print("-" * 47)
    for size in image_sizes:
        shape = (size, size)
        M = create_convolution_matrix(shape, kernel).astype(np.float64)
        det = np.linalg.det(M)
        cond = np.linalg.cond(M)
        print(f"{size}x{size:<12} {det:<15.4e} {cond:<15.4e}")
        if size < 4:
          print(M)


--- ANÁLISIS CON KERNEL NO NORMALIZADO (Suma = 16) ---
Tamaño Imagen   Determinante    Nº Condición    
-----------------------------------------------
2x2            8.1000e+01      9.0000e+00     
[[4. 2. 2. 1.]
 [2. 4. 1. 2.]
 [2. 1. 4. 2.]
 [1. 2. 2. 4.]]
3x3            4.0960e+03      3.3971e+01     
[[4. 2. 0. 2. 1. 0. 0. 0. 0.]
 [2. 4. 2. 1. 2. 1. 0. 0. 0.]
 [0. 2. 4. 0. 1. 2. 0. 0. 0.]
 [2. 1. 0. 4. 2. 0. 2. 1. 0.]
 [1. 2. 1. 2. 4. 2. 1. 2. 1.]
 [0. 1. 2. 0. 2. 4. 0. 1. 2.]
 [0. 0. 0. 2. 1. 0. 4. 2. 0.]
 [0. 0. 0. 1. 2. 1. 2. 4. 2.]
 [0. 0. 0. 0. 1. 2. 0. 2. 4.]]
4x4            3.9062e+05      8.9721e+01     
5x5            6.0466e+07      1.9399e+02     
25x25           5.6062e+70      7.4696e+04     
50x50           5.7150e+170     1.1098e+06     

--- ANÁLISIS CON KERNEL NORMALIZADO (Suma = 1) ---
Tamaño Imagen   Determinante    Nº Condición    
-----------------------------------------------
2x2            1.2360e-03      9.0000e+00     
[[0.25   0.125  0.125  0.0625]
 [0.

#### Pero, ¿por qué esto amplifica solamente el ruido y no toda la señal?

Recordemos que la imagen de salida que medimos en el mundo real, $Y$, está compuesta por dos partes: la señal ideal y el ruido.
$$Y = Y_{\text{señal}} + \text{Ruido}$$
donde sabemos que $Y_{\text{señal}} = M \cdot X$.

Cuando aplicamos la inversa para recuperar $X$, estamos haciendo:
$$X_{\text{recuperada}} = M^{-1} \cdot Y = M^{-1} \cdot (M \cdot X + \text{Ruido})$$
Aplicando la propiedad distributiva, separamos la operación en dos partes:
$$X_{\text{recuperada}} = (M^{-1} \cdot M \cdot X) + (M^{-1} \cdot \text{Ruido}) = X + (M^{-1} \cdot \text{Ruido})$$

Para verlo con la fórmula del determinante, hay una identidad matemática clave: $\text{Adj}(M) \cdot M = \det(M) \cdot I$.
$$M^{-1} \cdot (M \cdot X) = \left( \frac{1}{\det(M)} \cdot \text{Adj}(M) \right) \cdot (M \cdot X) = \frac{1}{\det(M)} \cdot (\text{Adj}(M) \cdot M) \cdot X$$
$$= \frac{1}{\det(M)} \cdot (\det(M) \cdot I) \cdot X = 1 \cdot I \cdot X = X$$


#2. DOMINIO DE LA FRECUENCIA

No es posible recuperar la imagen original porque, al tratarse de un filtro paso bajo, hay frecuencias altas que se _eliminan_. Es decir, debido al suavizado, hay frecuencias altas en la imagen resultante que, directamente, dejan de existir. Podríamos emplear lo que se llama **DECONVOLUCIÓN**, pero a nivel práctico no suele ser posible recuperar la imagen original a la perfección. Véase este vídeo de Shree Nayar (cuyo visionado os recomiendo): [Deconvolution | Image Processing II](https://www.youtube.com/watch?v=f-IINpceX6k).

Teóricamente, si la operación fuera puramente matemática y sin imperfecciones, la deconvolución sería una simple inversión. Sin embargo, en el mundo real, hay varios problemas fundamentales:



### 2.1. Pérdida irreversible de información
La forma más sencilla de entender la deconvolución es en el dominio de la frecuencia. La convolución ($f * h = g$, en donde $f$ es la imagen original, $h$ el kernel, y $g$ la imagen resultante) se convierte en una simple multiplicación (tras aplicar la transformada de Fourier a cada una de las señales anteriores):
$$ F \cdot H = G $$
Para recuperar la imagen original $F$, bastaría con dividir:
$$ F = \frac{G}{H} $$
El problema es que los kernels de suavizado ($h$) son filtros paso bajo. Esto significa que su transformada de Fourier ($H$) tiene valores de cero o muy cercanos a cero para las altas frecuencias (los detalles finos, los bordes). Al intentar dividir por cero, tenemos un problema: hemos perdido irreversiblemente esa información de alta frecuencia, dado que el suavizado la ha borrado.

Es importante remarcar que, matemáticamente hablando, si suponemos un kernel Gaussiano ideal, con precisión infinita, y no existiese ningún tipo de ruido, sí se podría recuperar la imagen original (porque $H(u,v)$ nunca sería exactamente cero). Pero, en la práctica digital, suele ser imposible recuperar perfectamente la imagen original, dado que los valores pequeños y nulos de $H(u,v)$ hacen la deconvolución numéricamente inestable. A efectos prácticos, esas frecuencias se pierden, como si estuvieran “eliminadas”.



### 2.2. El problema del ruido
El modelo real de una imagen no es $g = f * h$, sino:
$$ g = (f * h) + \text{ruido} $$
Cuando intentamos deconvolucionar esto en el dominio de la frecuencia, la ecuación se convierte en:
$$ F_{\text{recuperada}} = \frac{G}{H} = \frac{F \cdot H + \text{Ruido}}{H} = F + \frac{\text{Ruido}}{H} $$
El problema es que, en las frecuencias donde $H$ es muy pequeño, estamos dividiendo el ruido por un número diminuto. Esto amplifica el ruido de forma masiva, hasta el punto de que la imagen recuperada puede quedar completamente dominada por el ruido, como se puede ver en el ejemplo mostrado en el vídeo anterior ([Deconvolution | Image Processing II](https://www.youtube.com/watch?v=f-IINpceX6k)).



### 2.3. El kernel (h) suele ser una mera aproximación
Todo el proceso asume que conocemos a la perfección el kernel de desenfoque $h(x,y)$ (conocido como *Point Spread Function* o *PSF*). En una situación real (una foto movida, por ejemplo), casi nunca conocemos la *PSF* exacta. Si tu estimación del kernel es incorrecta, la deconvolución producirá artefactos extraños (como anillos o ecos alrededor de los bordes).

### Si tenemos una imagen de $N \times N$, conocemos el padding para obtener una salida de $N \times N$ y no hay ruido, entonces tenemos un sistema de $N^2$ ecuaciones con $N^2$ incógnitas que es teóricamente resoluble. ¿Cómo puede ser esto si, según acabamos de comentar, hemos "eliminado frecuencias"?

El grado de "eliminación" de esas frecuencias y, por tanto, la dificultad de la recuperación, depende directamente del tamaño ($N$) de la imagen. En una imagen pequeña ($2 \times 2$ o $3 \times 3$), la rejilla de píxeles es muy gruesa. Solo puede representar un número muy limitado de frecuencias. El filtro Gaussiano $H(u,v)$ atenúa estas frecuencias pero, como no son muy altas, el valor de $H$ sigue estando lejos de ser cero para las frecuencias presentes en la imagen. Esto se traduce, algebraicamente, en que la matriz $M$ es invertible y bien condicionada. En cierta medida, podríamos afirmar que, al tener pocos píxeles (en matrices $2 \times 2$ o $3 \times 3$), no puedes ni representar las frecuencias que el filtro destruiría, de modo que $M$ resulta invertible.

En una imagen de tamaño realista (ej. $512 \times 512$), la rejilla de píxeles es muy fina y puede representar frecuencias espaciales más altas. Para estas frecuencias tan altas, el valor del filtro Gaussiano $H(u,v)$ es prácticamente cero. La información contenida en estas frecuencias queda destruida. Esto se traduce en que la matriz $M$ (que sería de $262144 \times 262144$) es extremadamente mal condicionada. Es decir, aunque teóricamente sigues teniendo un sistema de $N^2 \times N^2$ ecuaciones, este sistema es tan inestable (mal condicionado) que es "inútil" en la práctica. El más mínimo ruido o error de redondeo es amplificado masivamente al intentar resolverlo.

En cualquier caso, teóricamente, si la salida y la entrada tienen la misma dimensión, conoces el padding y el kernel, no hay ruido ni errores numéricos, y la matriz $M$ es invertible y bien condicionada, entonces sí, puedes recuperar exactamente la imagen original previa al suavizado.

#CONCLUSIÓN

En general, no podemos recuperar la imagen/señal original a la perfección. La única opción sería disponer de una serie de *assumptions* y/o información extra de la que, en muchas ocasiones, carecemos. Es decir, salvo en casos de juguete y con información adicional, no es posible recuperar la imagen original. Otra posibilidad sería buscar una aproximación a la imagen original, y decidir, por ejemplo, emplear un filtro de realce (como *unsharp masking*) para resaltar ciertos detalles en la imagen suavizada (y, como consecuencia, aproximarnos a lo que podría haber sido la imagen original).